# 4. 임베딩과 검색 테스트

**목표:** 이미 저장된 청크의 텍스트를 임베딩하고 질문으로 Top-5를 찾은 뒤 부모 섹션과 관련 그림을 조회합니다. LLM 답변 생성은 다음 노트북에서 연결합니다.

순서: DB 자료 확인 → 대표 6개 검토 → 소량 임베딩·UPDATE → 질문 검색 → 부모 중복 제거·문맥 구성 → 필요 시 전체 임베딩.

수업의 `OpenAIEmbeddings`, `PGEngine`, `PGVectorStore`를 사용합니다. 기존 청크가 있으므로 새 테이블 생성이나 `add_documents()`를 호출하지 않습니다. 벡터는 기존 행에 `UPDATE`로 채우며 부모는 임베딩하지 않습니다.

**준비:** 프로젝트 `.venv` 커널, `.env`의 `DB_URL`과 `OPENAI_API_KEY`. 키는 노트북·채팅에 붙여 넣지 말고 로컬 `.env`에 설정하세요. 키가 없어도 1~3단계의 DB 조회와 자료 검토는 가능합니다. 5단계부터는 청크/질문 텍스트를 OpenAI 임베딩 API에 전송하며 사용량에 따른 비용이 발생합니다.

수업 모델은 `text-embedding-3-small`, 이 노트북의 차원 설정은 1536입니다. [OpenAI 공식 임베딩 문서](https://developers.openai.com/api/docs/guides/embeddings)를 참고했습니다.

In [ ]:
# 경로·환경변수·표 출력에 필요한 도구입니다.
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from psycopg.conninfo import conninfo_to_dict
from sqlalchemy.engine import make_url
from IPython.display import display
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGEngine, PGVectorStore

project_root = Path('/Users/anna/mle-02-p1-team1')
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))
from car_search_rag.common.database_manager import DatabaseManager

# 이 프로젝트의 설정을 읽고 접속 정보 자체는 출력하지 않습니다.
load_dotenv(project_root / '.env', override=False)
connection_info = conninfo_to_dict(os.environ['DB_URL'])
project_ref = 'hbuvewommtcmbbynsvht'
assert project_ref in connection_info.get('host', '') or project_ref in connection_info.get('user', ''), '팀 DB 주소를 확인하세요.'
os.environ.setdefault('PGCONNECT_TIMEOUT', '10')
db = DatabaseManager()
print('DB 연결 도구 준비 완료')
print('OpenAI API 키 설정 여부:', bool(os.getenv('OPENAI_API_KEY')))

## 1. 대상 설명서와 임베딩 모델 지정

모델명과 숫자 개수는 저장할 때도, 질문을 검색할 때도 같아야 합니다. 설정을 바꿔도 기존 벡터가 자동으로 바뀌지는 않습니다. 이미 다른 모델의 벡터가 있으면 중단하도록 구현합니다.

In [ ]:
# 3번에서 적재한 설명서입니다. 다른 PDF를 사용하면 DB의 document_id를 확인해 바꿉니다.
document_id = 'NE1_2027_ko_KR_4525f59eb857'
embedding_model = 'text-embedding-3-small'
embedding_dimension = 1536
batch_size = 64
top_k = 5

# embedder는 키가 필요할 때만 생성합니다. 아래 DB 조회는 API 키 없이 가능합니다.
embeddings = None

def get_embeddings():
    global embeddings
    if embeddings is None:
        # .env에 키를 추가한 뒤 이 함수를 다시 실행하면 읽을 수 있습니다.
        load_dotenv(project_root / '.env', override=False)
        if not os.getenv('OPENAI_API_KEY'):
            raise RuntimeError('프로젝트 .env에 OPENAI_API_KEY를 설정한 뒤 이 셀을 다시 실행하세요. 키를 출력하지 마세요.')
        embeddings = OpenAIEmbeddings(model=embedding_model, dimensions=embedding_dimension,
                                      max_retries=2, request_timeout=60)
    return embeddings

# 이 함수는 현재 문서의 임베딩 진행률만 읽습니다.
def embedding_status():
    with db.connect(camel_case_keys=False) as conn:
        row = conn.execute("""
            SELECT count(*) AS total,
                   count(*) FILTER (WHERE embedding IS NULL) AS pending,
                   count(*) FILTER (WHERE embedding IS NOT NULL
                       AND embedding_model=%s AND embedding_dimension=%s) AS ready,
                   count(*) FILTER (WHERE embedding IS NOT NULL
                       AND (embedding_model IS DISTINCT FROM %s OR embedding_dimension IS DISTINCT FROM %s)) AS incompatible
            FROM anna_rag.chunks WHERE document_id=%s
        """, (embedding_model, embedding_dimension, embedding_model, embedding_dimension, document_id)).fetchone()
    if not row['total']:
        raise ValueError('이 문서의 청크가 없습니다. 3번 적재 결과를 확인하세요.')
    if row['incompatible']:
        raise ValueError('다른 모델·차원의 벡터가 있습니다. 자동으로 덮어쓰지 않습니다.')
    return row

status = embedding_status()
display(pd.DataFrame([status]))
print('total=전체 청크 / pending=미처리 / ready=이 모델로 검색 가능 / incompatible=다른 모델')

## 2. 텍스트와 이미지 설명을 소량 검토

텍스트 3개와 이미지 설명 3개를 선택합니다. 비상 충전 해제 관련 내용을 우선하되 이 표본만으로 전체 검색 품질을 평가할 수는 없습니다. `caption_source=surrounding_text`는 주변 본문을 가져온 설명으로, 그림을 AI가 해석한 결과가 아닙니다.

In [ ]:
with db.connect(camel_case_keys=False) as conn:
    sample_rows = conn.execute("""
        WITH candidates AS (
            SELECT c.chunk_id, c.chunk_type, c.page_content, c.pdf_pages,
                   s.title, i.caption_source,
                   row_number() OVER (PARTITION BY c.chunk_type ORDER BY
                       CASE WHEN 18=ANY(c.pdf_pages) THEN 0 ELSE 1 END,
                       c.chunk_id) AS sample_order
            FROM anna_rag.chunks c
            JOIN anna_rag.sections s USING(document_id, section_id)
            LEFT JOIN anna_rag.images i USING(document_id, image_id)
            WHERE c.document_id=%s
        )
        SELECT * FROM candidates WHERE sample_order<=3 ORDER BY chunk_type, sample_order
    """, (document_id,)).fetchall()
sample_ids = [row['chunk_id'] for row in sample_rows]
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(pd.DataFrame(sample_rows)[['chunk_id', 'chunk_type', 'title', 'pdf_pages', 'caption_source', 'page_content']])
print('소량 시험 대상:', len(sample_ids), '개')

## 3. API 입력량 확인

토큰은 모델이 글을 처리하는 단위입니다. 글자 수와 같지 않습니다. `tiktoken`으로 미처리 청크의 입력량을 먼저 확인합니다. 이 셀은 임베딩 API를 호출하지 않습니다. 첫 실행 시 토크나이저 자료를 다운로드할 수 있습니다.

In [ ]:
# 수업 모델의 토크나이저입니다. 토큰 수를 세고 이후 부모 문맥 예산에도 사용합니다.
encoding = tiktoken.get_encoding('cl100k_base')

def pending_rows(chunk_ids=None):
    # sample_ids를 넘기면 소량만, None이면 문서 전체의 미처리 행을 읽습니다.
    query = 'SELECT chunk_id, page_content FROM anna_rag.chunks WHERE document_id=%s AND embedding IS NULL'
    params = [document_id]
    if chunk_ids is not None:
        query += ' AND chunk_id=ANY(%s)'
        params.append(chunk_ids)
    query += ' ORDER BY chunk_id'
    with db.connect(camel_case_keys=False) as conn:
        return conn.execute(query, params).fetchall()

pending = pending_rows()
token_lengths = [len(encoding.encode(row['page_content'], disallowed_special=())) for row in pending]
if token_lengths and max(token_lengths) > 8191:
    raise ValueError('너무 긴 청크가 있습니다. 다시 나눈 뒤 임베딩하세요.')
print('전체 미처리 청크:', len(pending))
print('예상 입력 토큰:', sum(token_lengths), '/ 가장 긴 청크:', max(token_lengths, default=0))
print('소량 시험 미처리 청크:', len(pending_rows(sample_ids)))

## 4. 임베딩 결과를 기존 행에 저장하는 함수

`embed_documents()`로 여러 텍스트를 벡터로 변환하고 `UPDATE`로 임베딩·모델명·차원을 함께 채웁니다. 이미 처리한 행은 건너뜁니다. 64개씩 저장하므로 중간에 실패하면 성공한 배치 다음부터 재실행할 수 있습니다.

API 호출 후 본문이 바뀌었다면 잘못된 벡터를 저장하지 않도록 중단합니다. 이 경우 해당 배치의 DB 변경은 취소됩니다. API 사용량은 DB 롤백으로 취소되지 않습니다. 본문을 나중에 수정할 때도 임베딩 3개 항목을 함께 NULL로 초기화한 뒤 다시 생성해야 합니다.

In [ ]:
def embed_and_save(chunk_ids=None):
    # 다른 모델의 벡터가 섞이지 않는지 먼저 확인합니다.
    embedding_status()
    rows = pending_rows(chunk_ids)
    if not rows:
        print('처리할 청크가 없습니다. 기존 임베딩을 재사용합니다.')
        return 0
    model = get_embeddings()
    saved = 0
    for offset in range(0, len(rows), batch_size):
        batch = rows[offset:offset + batch_size]
        texts = [row['page_content'] for row in batch]
        # 빈 글과 지나치게 긴 입력을 API 호출 전에 검사합니다.
        if any(not text.strip() or len(encoding.encode(text, disallowed_special=())) > 8191 for text in texts):
            raise ValueError('비어 있거나 너무 긴 청크를 확인하세요.')
        vectors = model.embed_documents(texts)
        if len(vectors) != len(batch):
            raise ValueError('요청한 텍스트 수와 받은 벡터 수가 다릅니다.')
        arrays = [np.asarray(vector, dtype=np.float32) for vector in vectors]
        if any(vector.shape != (embedding_dimension,) or not np.isfinite(vector).all()
               or np.linalg.norm(vector) == 0 for vector in arrays):
            raise ValueError('벡터의 차원·숫자·길이를 확인하세요. DB에 저장하지 않았습니다.')
        # API 호출 동안 DB 연결을 붙잡지 않고 결과를 받은 뒤 저장합니다.
        with db.connect(camel_case_keys=False) as conn:
            for row, vector in zip(batch, arrays):
                updated = conn.execute("""
                    UPDATE anna_rag.chunks
                    SET embedding=%s, embedding_model=%s, embedding_dimension=%s
                    WHERE document_id=%s AND chunk_id=%s AND embedding IS NULL AND page_content=%s
                    RETURNING chunk_id
                """, (vector, embedding_model, embedding_dimension,
                      document_id, row['chunk_id'], row['page_content'])).fetchone()
                if updated is None:
                    raise RuntimeError('처리 중 본문이나 벡터가 변경되었습니다. 이번 배치를 취소했습니다. 다시 조회하세요.')
        saved += len(batch)
        print(f'저장 완료: {saved}/{len(rows)}', flush=True)
    return saved

## 5. 대표 6개만 실제 임베딩

**이 셀은 OpenAI API를 호출하고 실제 DB에 벡터를 저장합니다.** 키가 없다면 여기서 설정 안내와 함께 멈춥니다. 소량 시험 후의 검색은 임베딩된 표본 안에서만 이루어집니다.

In [ ]:
# 새 행을 INSERT하지 않고 선택한 기존 청크만 UPDATE합니다.
sample_saved = embed_and_save(sample_ids)
print('이번에 새로 임베딩한 수:', sample_saved)
display(pd.DataFrame([embedding_status()]))

## 6. 수업의 PGVectorStore를 기존 테이블에 연결

수업에서는 기본 컬럼 이름으로 새 테이블을 만들었지만 여기서는 기존 `anna_rag.chunks`를 연결합니다. `id_column='chunk_id'`, `content_column='page_content'`로 실제 열 이름을 알려줍니다. 연결 자체는 청크를 다시 저장하지 않습니다.

In [ ]:
# 같은 DB_URL을 SQLAlchemy의 psycopg 연결 형식으로 변환합니다. 주소를 출력하지 않습니다.
engine_url = make_url(os.environ['DB_URL']).set(drivername='postgresql+psycopg')
engine = PGEngine.from_connection_string(url=engine_url)
store = PGVectorStore.create_sync(
    engine=engine, embedding_service=get_embeddings(),
    schema_name='anna_rag', table_name='chunks',
    id_column='chunk_id', content_column='page_content', embedding_column='embedding',
    metadata_columns=['document_id', 'section_id', 'chunk_type', 'pdf_pages', 'image_id',
                      'embedding_model', 'embedding_dimension'],
)
print('기존 chunks 테이블 연결 완료')

## 7. 질문을 임베딩하고 Top-5 검색

`embed_query()`는 질문 하나를 임베딩합니다. 저장된 문서와 같은 모델·차원을 사용해야 합니다. 반환 점수는 코사인 **거리**로 작을수록 가깝습니다. 정답 확률이 아닙니다. 문서 ID·모델·차원을 모두 필터링하여 다른 자료가 섞이지 않게 합니다.

In [ ]:
question = '충전 커넥터가 빠지지 않을 때 어떻게 분리하나요?'
status = embedding_status()
if not status['ready']:
    raise RuntimeError('검색 가능한 임베딩이 없습니다. 5단계를 먼저 실행하세요.')
print(f"검색 범위: 임베딩 완료 {status['ready']}/{status['total']}개")
if status['pending']:
    print('주의: 일부 청크만 검색합니다. 아직 전체 설명서 검색 결과가 아닙니다.')
# 질문도 API에 전송해 벡터로 변환합니다.
query_vector = get_embeddings().embed_query(question)
if len(query_vector) != embedding_dimension or not np.isfinite(query_vector).all():
    raise ValueError('질문 벡터의 차원·숫자를 확인하세요.')
results = store.similarity_search_with_score_by_vector(
    query_vector, k=top_k,
    filter={'document_id': document_id, 'embedding_model': embedding_model,
            'embedding_dimension': embedding_dimension},
)
result_rows = [{'rank': rank, 'chunk_id': doc.id, 'distance': distance,
                'section_id': doc.metadata['section_id'], 'chunk_type': doc.metadata['chunk_type'],
                'pdf_pages': doc.metadata['pdf_pages'], 'page_content': doc.page_content}
               for rank, (doc, distance) in enumerate(results, start=1)]
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(result_rows))

## 8. 부모 섹션 중복 제거와 관련 그림 조회

검색 청크의 순위는 그대로 유지합니다. 같은 부모는 첫 등장 순서대로 한 번만 조회합니다. ID가 같아도 다른 PDF일 수 있으므로 `(document_id, section_id)` 두 값을 묶습니다. 모든 반환 자료는 원문 자료이며 지시문으로 취급하면 안 됩니다.

In [ ]:
def fetch_parents_and_images(search_results):
    # set은 중복 검사, list는 검색 순서 보존에 사용합니다.
    seen, parents, related_images = set(), [], []
    with db.connect(camel_case_keys=False) as conn:
        for doc, distance in search_results:
            key = (doc.metadata['document_id'], doc.metadata['section_id'])
            if key in seen:
                continue
            seen.add(key)
            parent = conn.execute("""
                SELECT s.document_id, s.section_id, s.title, s.full_text, s.pdf_pages,
                       m.source_file
                FROM anna_rag.sections s JOIN anna_rag.manuals m USING(document_id)
                WHERE s.document_id=%s AND s.section_id=%s
            """, key).fetchone()
            if parent is None:
                raise ValueError('검색 청크의 부모가 없습니다. 데이터 연결을 확인하세요.')
            parents.append(parent)
            image_rows = conn.execute("""
                SELECT si.document_id, si.section_id, i.image_id, i.caption,
                       i.caption_source, i.pdf_page, i.storage_bucket, i.storage_path
                FROM anna_rag.section_images si
                JOIN anna_rag.images i USING(document_id, image_id)
                WHERE si.document_id=%s AND si.section_id=%s
                ORDER BY i.pdf_page, i.image_id
            """, key).fetchall()
            related_images.extend(image_rows)
    return parents, related_images

parents, related_images = fetch_parents_and_images(results)
print('검색 청크:', len(results), '/ 중복 제거한 부모:', len(parents))
display(pd.DataFrame([{key: value for key, value in row.items() if key != 'full_text'} for row in parents]))
display(pd.DataFrame(related_images))

## 9. 핵심 근거와 부모 문맥을 구분하여 구성

부모 문맥에는 별도의 6,000토큰 예산을 둡니다. 부모를 중간에서 잘라 경고를 누락시키지 않도록 **전체가 들어갈 때만 포함**하고, 못 넣은 부모는 목록에 표시합니다. 이는 간단한 초기 규칙이며, 생략된 부모의 조건이 중요할 수 있으므로 그대로 최종 답변을 생성하지 말고 범위·예산을 검토해야 합니다.

예산은 부모 문맥만의 상한입니다. 질문·검색 청크·안내문과 향후 답변에 필요한 토큰은 별도로 계산해야 합니다. 이 셀은 LLM을 호출하지 않습니다.

In [ ]:
def build_context(search_results, parent_rows, parent_token_budget=6000):
    evidence_blocks = []
    for rank, (doc, distance) in enumerate(search_results, start=1):
        evidence_blocks.append(f"[핵심 근거 후보 {rank}] chunk_id={doc.id} / PDF {doc.metadata['pdf_pages']} / 거리={distance:.4f}\n{doc.page_content}")
    included, omitted = [], []
    for parent in parent_rows:
        block = (f"[부모 보조 문맥] {parent['title']} / {parent['source_file']} / PDF {parent['pdf_pages']}\n"
                 f"document_id={parent['document_id']} / section_id={parent['section_id']}\n{parent['full_text']}")
        candidate = '\n\n'.join(included + [block])
        # 구분자까지 포함한 실제 합친 문자열의 토큰 수를 계산합니다.
        if len(encoding.encode(candidate, disallowed_special=())) <= parent_token_budget:
            included.append(block)
        else:
            omitted.append({'document_id': parent['document_id'], 'section_id': parent['section_id'],
                            'title': parent['title'], 'reason': '부모 문맥 토큰 예산 초과'})
    evidence = '\n\n'.join(evidence_blocks)
    parent_context = '\n\n'.join(included)
    return {'evidence': evidence, 'parent_context': parent_context, 'omitted_parents': omitted,
            'parent_tokens': len(encoding.encode(parent_context, disallowed_special=())),
            'evidence_tokens': len(encoding.encode(evidence, disallowed_special=()))}

context = build_context(results, parents)
print('핵심 근거 토큰:', context['evidence_tokens'], '/ 부모 문맥 토큰:', context['parent_tokens'])
print('예산 때문에 제외한 부모:', context['omitted_parents'])
print(context['evidence'])
print('\n===== 부모 보조 문맥 =====\n')
print(context['parent_context'])

## 10. 검토 후 전체 미처리 청크로 확대

대표 자료와 검색 연결을 확인한 뒤 `RUN_FULL_EMBEDDING=True`로 바꾸어 실행합니다. 현재 값 False는 노트북을 처음 전체 실행했을 때 소량 시험만 하도록 하는 학습 설정입니다. 기존에 같은 모델로 저장한 벡터는 재사용합니다.

전체 저장 후에는 **7~9단계 검색 셀을 다시 실행**해야 결과가 갱신됩니다. 사용한 질문이 같으면 이미 생성한 질문 벡터를 재사용하는 방식으로 확장할 수도 있습니다.

In [ ]:
RUN_FULL_EMBEDDING = True
if RUN_FULL_EMBEDDING:
    total_saved = embed_and_save()
    print('이번에 추가한 임베딩:', total_saved)
    display(pd.DataFrame([embedding_status()]))
else:
    print('소량 시험 모드입니다. 검토 후 True로 바꾸면 전체 미처리 청크를 임베딩합니다.')

## 실행 결과를 읽는 방법

- `ready=6`이라면 6개에서만 검색합니다. 전체 2,338개를 검색했다고 해석하지 않습니다.
- 이미지 검색은 캡션 텍스트의 임베딩 검색입니다. 그림의 픽셀을 이해하는 모델은 사용하지 않습니다.
- Top-5가 반환돼도 문서에 답이 있다는 보장은 없습니다. 다음 답변 생성 단계에서 근거 부족·조건 충돌을 처리해야 합니다.
- 이 노트북에서는 테이블 삭제, 재청킹, 이미지 재업로드를 하지 않습니다. 전체 임베딩 후에도 부모/이미지 연결은 유지됩니다.
- 다음 단계는 `context['evidence']`와 `context['parent_context']`의 역할을 구분해 LLM에 전달하고 답변·출처·이미지를 표시하는 것입니다.

## 이번 작성 시 실행 확인 (2026-10-05)
대표 청크 6개를 실제 임베딩했고 Top-5 검색 및 부모·이미지 조회를 실행했습니다. 전체 2,338개 중 6개 완료, 2,332개 미처리 상태입니다.
질문 “충전 커넥터가 빠지지 않을 때 어떻게 분리하나요?”의 표본 검색 1위는 18페이지 본문이었습니다. 서로 다른 부모 2개를 조회했고, 부모 문맥은 3,072토큰으로 예산 안에 포함됐습니다. 이는 표본 연결 확인이며 전체 검색 품질 평가 결과가 아닙니다.
저장 함수의 재실행 건너뛰기·벡터 차원 검사·행 수 유지도 검증했습니다. 검증용 가짜 벡터는 별도 트랜잭션에서 롤백했으며 DB에는 실제 API가 생성한 벡터 6개만 남았습니다.
